# Agent 구성

- 올림픽 정보 제공: Qdrant DB와 연동하는 **임베딩 벡터 Retrieve Tool**
- 영화 정보 제공: Neo4j 연동 **GraphRAG Tool**


In [1]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

from qdrant_client import QdrantClient

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
################################################
# QDrant-info-wiki Collection 연동 tool
################################################
from langchain_core.tools.retriever import create_retriever_tool

COLLECTION_NAME = "olympic_info_wiki"
VECTOR_SIZE = 3072

client = QdrantClient(url="http://localhost:6333")

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = QdrantVectorStore(
    client=client,
    embedding=embeddings, 
    collection_name=COLLECTION_NAME 
    # ,vector_name="dense" => dense, sparse 있으면 이름 정해줘야 함
)

# VectorStore 를 Retriever로 변환
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 7}
)

# Tool로 변환 
search_olympic_info_tool = create_retriever_tool(
    retriever=retriever,
    name="search_olympic_info",
    description="VectorDB의 Olympic 정보를 유사도 검색을 통해 조회하는 도구"
)

In [ ]:
#####################################################################
# 영화 정보를 제공하는 GraphDB 연동 RAG tool
#####################################################################
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain
from langchain_neo4j import Neo4jGraph
from langchain_neo4j.chains.graph_qa.cypher import CYPHER_GENERATION_PROMPT

# cypher search 쿼리를 실행하지 못한다.
custom_prompt_str = CYPHER_GENERATION_PROMPT.template+\
"""
<constraint>
- 검색조건이 영화제목, 감독이나 배우이름일 경우 입력값을 영어로 변경해서 검색한다. (예: "영화 '인셉션'의 감독과 개봉일을 알려줘." ->" 영화 'Inception'의 감독과 개봉일을 알려줘.")
- Cypher 쿼리는 NEO4J 5.0 문법에 맞게 작성한다.
</constraint>
"""

graph = Neo4jGraph()
prompt_template = PromptTemplate(template=custom_prompt_str)
cypher_chain = GraphCypherQAChain.from_llm( # 유사도 검색 불가. 관계만 가능(유사도 조회 필요 시 RAG tool(쿼리-서치 등) 만들어 사용 -> 쿼리가 유사도를 찾는다면 그 tool을 쓰라고 알려줌.)
    llm=ChatOpenAI(model="gpt-5.5", temperature=0.0), 
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    cypher_prompt=prompt_template
)

In [4]:
query = "크리스토퍼 놀란의 영화에 가장 많이 출연한 배우는 누구인가?"
cypher_chain.invoke({"query": query})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:Person {name: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person)
RETURN a.name AS actor, count(DISTINCT m) AS appearances
ORDER BY appearances DESC
LIMIT 1;
Full Context:
[{'actor': 'Michael Caine', 'appearances': 5}]

> Finished chain.


{'query': '크리스토퍼 놀란의 영화에 가장 많이 출연한 배우는 누구인가?',
 'result': '크리스토퍼 놀란의 영화에 가장 많이 출연한 배우는 Michael Caine이며, 총 5번 출연했습니다.'}

In [5]:
#################################
# TOOL 로 변환
#################################
@tool
def search_movie_info_tool(query: str) -> str:
    """NEO4J 에서 영화 정보를 검색하는 데 사용하는 도구입니다. 그래프 데이터베이스에 저장된 엔티티 간 관계, 구조적 질의(예: 'A와 연결된 B는?', '몇 개의 관계가 있는가')에 답할 때 사용하세요"""
    response = cypher_chain.invoke({"query": query})
    return response["result"]


# search_movie_info.invoke("크리스토퍼 놀란의 영화에 가장 많이 출연한 배우는 누구인가?")

In [6]:
######################################################
# Agent 생성 
# llm + search_movie_info, search_olympic_info_tool 
######################################################
from langchain.agents import create_agent

agent = create_agent(
    model=ChatOpenAI(model="gpt-5.5"),
    tools=[search_movie_info_tool, search_olympic_info_tool],
    system_prompt="""올림픽와 영화정보를 친절히 설명하는 유능한 Agent입니다. 
영화정보 검색을 위한 search_movie_info, 올림픽관련 정보를 검색하기 위한 search_olympic_info_tool 두개의 tool을 사용할 수 있습니다."""
)

In [13]:
# query = "톰 크루즈와 같은 영화에 출연한 배우들은 누가 있나?"
# query = "런던 올림픽이 개최된 년도와 그때 개봉한 영화는 뭐가 있나?"
query = "2차 세계대전을 배경으로 한 영화들을 추천해줘."

res = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie)
WHERE any(term IN ['world war ii', 'world war 2', 'wwii', 'ww2', 'second world war'] WHERE toLower(coalesce(m.title, '')) CONTAINS term OR toLower(coalesce(m.overview, '')) CONTAINS term OR toLower(coalesce(m.tagline, '')) CONTAINS term)
OPTIONAL MATCH (m)-[:IN_GENRE]->(g:Genre)
RETURN m.title AS title, m.released AS released, m.rating AS rating, m.runtime AS runtime, m.overview AS overview, collect(g.name) AS genres
ORDER BY m.rating DESC, m.released DESC
LIMIT 20
Full Context:
[{'title': "Schindler's List", 'released': neo4j.time.DateTime(1993, 11, 29, 0, 0, 0, 0), 'rating': 8.3, 'runtime': 195, 'overview': 'The true story of how businessman Oskar Schindler saved over a thousand Jewish lives from the Nazis while they worked as slaves in his factory during World War II.', 'genres': ['Drama', 'War', 'History']}, {'title': 'The Imitation Game', 'released': neo4j.time.DateTime(2014, 11, 14, 0, 0, 0, 0), 'ratin

In [14]:
res['messages']

[HumanMessage(content='2차 세계대전을 배경으로 한 영화들을 추천해줘.', additional_kwargs={}, response_metadata={}, id='7a0b7095-cbeb-4fa5-8a03-b377478a3d77'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 294, 'total_tokens': 356, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 22, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-E00lmQR7DW47CLWUEJeOvBhmqzrE1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f4b1d-b39b-7a51-ad02-c52408330122-0', tool_calls=[{'name': 'search_movie_info_tool', 'args': {'query': '2차 세계대전을 배경으로 한 영화 추천 목록 World War II movies'}, 'id': 'call_SbhEsIWrveJmUVIvVFrNo4oM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inpu

In [15]:
print(res['messages'][-1].content)

2차 세계대전을 배경으로 한 영화 추천작입니다.

1. **쉰들러 리스트**  
   나치 치하에서 유대인들을 구한 오스카 쉰들러의 실화를 바탕으로 한 작품입니다. 묵직한 역사 드라마를 원한다면 강력 추천합니다.

2. **이미테이션 게임**  
   제2차 세계대전 중 독일군 암호 ‘에니그마’를 해독하려 한 앨런 튜링과 암호 해독팀의 이야기입니다. 전쟁사와 천재 인물 드라마를 함께 볼 수 있습니다.

3. **피아니스트**  
   나치 점령하의 바르샤바에서 살아남으려는 유대인 피아니스트의 생존기를 그린 영화입니다. 전쟁의 참혹함을 매우 현실적으로 보여줍니다.

4. **바스터즈: 거친 녀석들**  
   나치 점령기의 프랑스를 배경으로 한 쿠엔틴 타란티노 감독의 대체역사 전쟁 영화입니다. 긴장감 있는 대사와 독특한 연출이 인상적입니다.

5. **특전 U보트 / Das Boot**  
   독일 잠수함 승무원들의 극한 상황을 그린 전쟁 영화입니다. 밀폐된 잠수함 안의 긴장감이 뛰어납니다.

6. **대탈주**  
   독일 포로수용소에서 연합군 포로들이 대규모 탈출을 계획하는 실화 기반 영화입니다. 고전 전쟁 영화의 대표작입니다.

7. **퓨리**  
   1945년 전쟁 막바지, 미군 전차 부대의 임무를 그린 영화입니다. 전차전과 전장의 압박감을 강하게 느낄 수 있습니다.

8. **우리 생애 최고의 해**  
   제2차 세계대전이 끝난 뒤 귀환한 참전 군인들이 사회에 적응해가는 과정을 그린 작품입니다. 전쟁 이후의 삶을 다룬 점이 특징입니다.

개인적으로는 처음 본다면 **쉰들러 리스트**, **피아니스트**, **이미테이션 게임**을 먼저 추천드립니다.  
전투 장면이 강한 영화를 원한다면 **퓨리**나 **Das Boot**, 독특한 스타일을 원한다면 **바스터즈: 거친 녀석들**이 좋습니다.
